In [1]:
%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = None
%sql sqlite:///../data/video_data.db
%config SqlMagic.feedback = 0

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

Connecting to 'sqlite:///../data/video_data.db'

In [2]:
# %sql DROP TABLE IF EXISTS videos;
# %sql DROP TABLE IF EXISTS channel_ids;

In [3]:
%%sql 
CREATE TABLE IF NOT EXISTS videos (
    kind TEXT,
    etag TEXT,
    id TEXT,
    snippet_publishedAt TEXT,
    snippet_channelId TEXT,
    snippet_title TEXT,
    snippet_description TEXT,
    snippet_thumbnails_default_url TEXT,
    snippet_thumbnails_default_width TEXT,
    snippet_thumbnails_default_height TEXT,
    snippet_thumbnails_medium_url TEXT,
    snippet_thumbnails_medium_width TEXT,
    snippet_thumbnails_medium_height TEXT,
    snippet_thumbnails_high_url TEXT,
    snippet_thumbnails_high_width TEXT,
    snippet_thumbnails_high_height TEXT,
    snippet_thumbnails_standard_url TEXT,
    snippet_thumbnails_standard_width TEXT,
    snippet_thumbnails_standard_height TEXT,
    snippet_thumbnails_maxres_url TEXT,
    snippet_thumbnails_maxres_width TEXT,
    snippet_thumbnails_maxres_height TEXT,
    snippet_channelTitle TEXT,
    snippet_tags TEXT,
    snippet_categoryId TEXT,
    snippet_liveBroadcastContent TEXT,
    snippet_localized_title TEXT,
    snippet_localized_description TEXT,
    snippet_defaultAudioLanguage TEXT,
    contentDetails_duration TEXT,
    contentDetails_dimension TEXT,
    contentDetails_definition TEXT,
    contentDetails_caption TEXT,
    contentDetails_licensedContent TEXT,
    contentDetails_projection TEXT,
    statistics_viewCount TEXT,
    statistics_likeCount TEXT,
    statistics_favoriteCount TEXT,
    statistics_commentCount TEXT,
    topicDetails_topicCategories TEXT,
    contentDetails_contentRating_ytRating TEXT,
    contentDetails_regionRestriction_blocked TEXT,
    timestamp TEXT,
    query TEXT
);

++
||
++
++

In [4]:
%%sql 
CREATE TABLE IF NOT EXISTS channel_ids (
    channel_id TEXT,
    written TEXT
);

++
||
++
++

In [5]:
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build

from tqdm.notebook import tqdm as tqdm

import pandas as pd

import sys

from sqlalchemy import create_engine

def get_channel_ids(query_string, api_keys):
    for api_key in api_keys:
        try:
            youtube = build('youtube', 'v3', developerKey=api_key)
            response = youtube.search().list(
                maxResults=50,
                part='snippet',
                q=query_string,
                relevanceLanguage='en'
            ).execute()
            return [item['snippet']['channelId'] for item in response['items']]
        except HttpError as e:
            if e.resp.status == 403:
                continue
            else:
                raise e
    raise Exception("All API keys have exceeded their quota.")

def get_all_channel_ids(query_string, api_keys, verbose=False):
    channel_ids = []
    next_page_token = None
    current_key_index = 0
    ix = 0
    while True:
        sys.stdout.write(f"\r{ix}")
        sys.stdout.flush()
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])
            
            response = youtube.search().list(
                maxResults=50,
                part='snippet',
                q=query_string,
                relevanceLanguage='en',
                pageToken=next_page_token
            ).execute()
            
            channel_ids.extend([item['snippet']['channelId'] for item in response['items']])
            
            next_page_token = response.get('nextPageToken')

            ix += 50
            
            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status in [403, 429]:  # Quota exceeded
                current_key_index += 1
                if current_key_index >= len(api_keys):
                    print("All API keys exhausted.")
                    break
                if verbose:
                    print(f"Switching to next API key. Current key index: {current_key_index}")
            else:
                raise  # Re-raise the exception if it's not a quota error
    return channel_ids
    
def get_playlist_video_ids(playlist_id, api_keys, verbose=False):
    video_ids = []
    next_page_token = None
    current_key_index = 0

    ix = 0

    while True:
        
        sys.stdout.write(f"\r{ix}")
        sys.stdout.flush()
        
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])
            
            playlist_items = youtube.playlistItems().list(
                part='contentDetails',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            ).execute()
            
            video_ids.append([item['contentDetails']['videoId'] for item in playlist_items['items']])
            
            next_page_token = playlist_items.get('nextPageToken')

            ix += 50

            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status in [403, 404, 429]:  # Quota exceeded
                current_key_index += 1
                if current_key_index >= len(api_keys):
                    print("All API keys exhausted.")
                    break
                if verbose:
                    print(f"Switching to next API key. Current key index: {current_key_index}")
            else:
                raise  # Re-raise the exception if it's not a quota error

    return video_ids

def get_video_response(video_id, api_keys):
    for api_key in api_keys:
        try:
            youtube = build('youtube', 'v3', developerKey=api_key)
            response = youtube.videos().list(
                part='snippet,contentDetails,statistics,topicDetails',
                id=video_id
            ).execute()
            return response
        
        except HttpError as e:
            if e.resp.status == 403:
                continue
            elif e.resp.status in [400, 404, 503]:
                return
            else:
                raise e
    raise Exception("All API keys have exceeded their quota.")

In [6]:
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()

# Search term
query_string = 'Elden Ring'
# Get channel IDs in database
db_channel_ids = %sql SELECT DISTINCT channel_id from channel_ids;
db_channel_ids = set(db_channel_ids.DataFrame()['channel_id'])
# Get channel IDs from search
query_channel_ids = set(get_all_channel_ids(query_string, API_KEYS))
# Get channel IDs from search that are not in database
use_channel_ids = query_channel_ids - db_channel_ids
# Get channel IDs as df
use_channel_id_df = pd.DataFrame([list(use_channel_ids), [False] * len(use_channel_ids)], index=['channel_id', 'written']).T
# Write channel IDs to database
db_loc = 'sqlite:///../data/video_data.db'
engine = create_engine(db_loc)
use_channel_id_df.to_sql('channel_ids', engine, if_exists='append', index=False)
read_channel_ids = %sql SELECT * from channel_ids;
read_channel_ids_df = read_channel_ids.DataFrame()

written_df = read_channel_ids_df[read_channel_ids_df['written'].astype(int).astype(bool)]
unwritten_df = read_channel_ids_df[~read_channel_ids_df['written'].astype(int).astype(bool)]

written_df.shape[0], unwritten_df.shape[0]

650

(0, 287)

In [7]:
channel_ids = %sql SELECT * from channel_ids;
channel_ids_df = channel_ids.DataFrame()
unwritten_df = channel_ids_df[~channel_ids_df['written'].astype(int).astype(bool)]
for channel_id in tqdm(unwritten_df['channel_id'], desc='Channel ID'):
    playlist_id = 'UU' + channel_id[2:]
    video_ids_lists = get_playlist_video_ids(playlist_id, API_KEYS)
    video_strings = [','.join(video_ids) for video_ids in video_ids_lists]
    video_responses = [get_video_response(video_id, API_KEYS) for video_id in tqdm(video_strings, desc=f'{channel_id} Video Response')]
    video_response_items = [response['items'] for response in tqdm(video_responses, desc=f'{channel_id} Video Response Items') if response]
    if video_response_items:
        video_response_df = pd.concat([pd.json_normalize(item) for item in tqdm(video_response_items, desc=f'{channel_id} Video Response Items')])
        video_response_df['timestamp'] = pd.Timestamp.now()
        video_response_df['query'] = query_string
        video_response_df.columns = [col.replace('.', '_') for col in video_response_df.columns]
        # Get DB columns
        db_loc = '../data/video_data.db'
        engine = create_engine(f'sqlite:///{db_loc}', echo=False)
        db_videos_columns = %sql SELECT * FROM videos LIMIT 1
        db_videos_columns = db_videos_columns.DataFrame().columns
        # if a column is in the database, but not in the dataframe, add it as a column of NaNs
        for col in tqdm(db_videos_columns, desc=f'{channel_id} Columns'):
            if col not in video_response_df.columns:
                video_response_df[col] = pd.NA
            video_response_df[col] = video_response_df[col].apply(lambda x: ','.join(x) if isinstance(x, list) else x)
        # if a column is not in the database, drop it
        video_response_df = video_response_df[[col for col in video_response_df.columns if col in db_videos_columns]]
        video_response_df.to_sql('videos', con=engine, if_exists='append', index=False)
        channel_ids_df.loc[channel_ids_df['channel_id'] == channel_id, 'written'] = True
        channel_ids_df.to_sql('channel_ids', con=engine, if_exists='replace', index=False)
        print(f'{channel_id} written to database')
    else:
        channel_ids_df.loc[channel_ids_df['channel_id'] == channel_id, 'written'] = True
        channel_ids_df.to_sql('channel_ids', con=engine, if_exists='replace', index=False)
    channel_ids_data = %sql SELECT * from channel_ids;
    channel_ids_df = channel_ids_data.DataFrame()

Channel ID:   0%|          | 0/287 [00:00<?, ?it/s]

0

UCIZrQ4rSsVlOedqjHtkOJgg Video Response:   0%|          | 0/1 [00:00<?, ?it/s]

UCIZrQ4rSsVlOedqjHtkOJgg Video Response Items:   0%|          | 0/1 [00:00<?, ?it/s]

UCIZrQ4rSsVlOedqjHtkOJgg Video Response Items:   0%|          | 0/1 [00:00<?, ?it/s]

...

UCRC6cNamj9tYAO6h_RXd5xA Video Response:   0%|          | 0/21 [00:00<?, ?it/s]

UCRC6cNamj9tYAO6h_RXd5xA Video Response Items:   0%|          | 0/21 [00:00<?, ?it/s]

UCRC6cNamj9tYAO6h_RXd5xA Video Response Items:   0%|          | 0/21 [00:00<?, ?it/s]

UCRC6cNamj9tYAO6h_RXd5xA Columns:   0%|          | 0/44 [00:00<?, ?it/s]

UCRC6cNamj9tYAO6h_RXd5xA written to database


In [8]:
# read the data table
video_data = %sql SELECT * FROM videos;
video_df = video_data.DataFrame()
print(video_df.shape)
print(video_df.drop_duplicates(subset=['id']).shape)

(389691, 44)
(389691, 44)


snippet_categoryId
24    10290
20     5379
22     4715
1       313
26      184
23      130
10      109
28        4
27        4
30        3
2         2
17        1
Name: count, dtype: int64